# 3 · Random forest classifier from scratch

Trains and tests a random forest that classifies detections (e.g. whistle contours) by their measured features, then classifies whole groups (encounters) by averaging the detection predictions.

**Input** — either:
- **ROCCA contour stats** exported by PAMGuard's ROCCA module (`RoccaContourStats*.csv`): a single file, a folder of them, or a folder hierarchy where folder names give labels (e.g. `species/location/encounter/`); or
- **any detection table** (CSV) with a label column, a group column and numeric feature columns.

**Output** in `OUTPUT_DIR`: detection predictions from cross-validation (standard classifier output format, usable for transfer learning), group predictions, accuracy summary, and the trained model (`.joblib`) for classifying new data.

By default the notebook runs on a small **synthetic** ROCCA-format example (`examples/RoccaContourStats_example_synthetic.csv`).

**How it works**
- *Training sample*: each group (encounter) contributes at most `MAX_PER_GROUP` random detections, so long encounters don't dominate. Optionally, the `PRUNE` share of each class's detections furthest from the class centre are dropped as outliers.
- *Balanced trees*: every tree is grown on the same number of detections from each class (the size of the smallest class), so common classes aren't favoured.
- *Testing*: leave-one-group-out — each group is predicted by a forest trained on all the others.
- *Decision score*: for a prediction with top probability p1 and runner-up p2, score = p1 × (p1 − p2). Low-scoring detections can be left out of group predictions (`MIN_SCORE`), and low-scoring groups can be discarded (`MIN_GROUP_SCORE`).

## Setup
Locally: `pip install -e "python[notebooks]"` from the repository folder. In Google Colab, run the next cell.

In [ ]:
# Google Colab only
# %pip install -q "speciesclassifiers @ git+https://github.com/tristankleyn/speciesClassifiers#subdirectory=python"

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from speciesclassifiers.plots import plot_confusion, plot_importance
from speciesclassifiers.randomforest import (RandomForestModel, RFParams, cross_validate, fit, predict,
                                             read_feature_table, read_rocca, summarise, threshold_curve)

## Load data

In [ ]:
# --- Data --------------------------------------------------------------------------------------
DATA_TYPE  = "rocca"      # "rocca" (RoccaContourStats CSVs) or "table" (any detection table CSV)
DATA_PATH  = "../../examples/RoccaContourStats_example_synthetic.csv"   # file or folder
VOC_TYPE   = "whistle"    # rocca only: "whistle" or "click"
LEVELS     = None         # rocca folder hierarchy: column per folder level, e.g. ["label", "location", "event_id"]
                          # None = labels from the CSV's KnownSpecies / EncounterID columns
LABEL_COL  = "label"      # table only: column with the class
GROUP_COL  = "event_id"   # independent unit for testing (encounter, location, ...)
OMIT_VARS  = None         # features to leave out, e.g. ["DURATION"]; None = defaults
FILTER_MIN = {}           # e.g. {"DURATION": 0.1, "FREQMEAN": 1000}: drop rows below
FILTER_MAX = {}           # e.g. {"DURATION": 5.0}: drop rows above
SUBSET     = {}           # keep only some values, e.g. {"label": ["Dde", "Ttr"], "location": ["Site A"]}
OUTPUT_DIR = "randomforest_run"

if DATA_TYPE == "rocca":
    data, features = read_rocca(DATA_PATH, VOC_TYPE, LEVELS, OMIT_VARS, FILTER_MIN, FILTER_MAX)
    LABEL_COL = "label"
else:
    data, features = read_feature_table(DATA_PATH, LABEL_COL, GROUP_COL, exclude=OMIT_VARS or ())
    for col, v in FILTER_MIN.items():
        data = data[data[col] >= v]
    for col, v in FILTER_MAX.items():
        data = data[data[col] <= v]
for col, keep in SUBSET.items():
    data = data[data[col].isin(keep)]
data[LABEL_COL] = data[LABEL_COL].astype(str)
out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
print(f"{len(data)} detections, {data[GROUP_COL].nunique()} groups, {len(features)} features")
data.groupby(LABEL_COL).agg(groups=(GROUP_COL, "nunique"), detections=(GROUP_COL, "size"))

Average of each feature per class:

In [ ]:
data.groupby(LABEL_COL)[features].mean().T.round(2)

## Explore
Scatter of two features (or a histogram of one), coloured by class.

In [ ]:
VARIABLE1 = features[0]   # e.g. "FREQBEG"
VARIABLE2 = features[1]   # e.g. "FREQEND"; None for a histogram of VARIABLE1

fig, ax = plt.subplots(figsize=(6, 4.5))
for lab, g in data.groupby(LABEL_COL):
    if VARIABLE2:
        ax.scatter(g[VARIABLE1], g[VARIABLE2], s=12, alpha=0.6, label=lab)
    else:
        ax.hist(g[VARIABLE1], bins=30, alpha=0.5, label=lab)
ax.set(xlabel=VARIABLE1, ylabel=VARIABLE2 or "Detections")
ax.legend(); plt.show()

## Train and test

In [ ]:
N_TREES       = 500     # trees in the forest
MTRY          = None    # features tried per split; None = square root of the number of features
NODE_SIZE     = 25      # minimum detections per leaf (larger = simpler trees)
MAX_PER_GROUP = 25      # most detections per group used for training
PRUNE         = 0.0     # share (0-1) of each class's training detections to drop as outliers
SEED          = 42
OMIT_GROUPS   = []      # groups to leave out entirely

params = RFParams(N_TREES, MTRY, NODE_SIZE, MAX_PER_GROUP, PRUNE, SEED)
use = data[~data[GROUP_COL].isin(OMIT_GROUPS)]
preds = cross_validate(use, features, LABEL_COL, GROUP_COL, params, classifier="randomforest", voc_type=VOC_TYPE)
preds.to_csv(out / "cv_detection_predictions.csv", index=False)

## Results
`MIN_SCORE` drops low-scoring detections before averaging; `MIN_GROUP_SCORE` discards low-scoring groups (more confident, fewer classified).

In [ ]:
MIN_SCORE = 0.02
MIN_GROUP_SCORE = 0.0

s = summarise(preds, MIN_SCORE, MIN_GROUP_SCORE, group_col="event_id")
s["groups"].to_csv(out / "cv_group_predictions.csv", index=False)
print(f"Group accuracy: {s['group_accuracy']:.1%}   Mean class accuracy: {s['mean_class_accuracy']:.1%}   "
      f"Groups discarded: {s['groups_discarded']:.1%}")
display(s["accuracy_by_class"].rename("accuracy").to_frame().style.format("{:.0%}"))
plot_confusion(s["confusion"], "Groups"); plt.show()

Groups in detail, misclassified first:

In [ ]:
g = s["groups"].assign(correct=lambda x: x["predicted"] == x["true_class"])
g[["event_id", "true_class", "predicted", "correct", "score", "n", "n_total"]].sort_values(["correct", "score"])

Accuracy against the group score threshold: how much accuracy you gain by discarding less confident groups.

In [ ]:
curve = threshold_curve(preds)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(curve["threshold"], curve["group_accuracy"], label="group accuracy")
ax.plot(curve["threshold"], curve["mean_class_accuracy"], label="mean class accuracy")
ax.plot(curve["threshold"], curve["groups_kept"], "--", label="groups kept")
ax.set(xlabel="Minimum group score", ylim=(0, 1.05)); ax.legend(); plt.show()

## Final model
Trained on all the data with the same settings, saved for classifying new data. Feature importance shows which measurements the forest relies on most.

In [ ]:
model = fit(use, features, LABEL_COL, GROUP_COL, params)
model.save(out / "randomforest_model.joblib")
(out / "summary.json").write_text(json.dumps({
    "params": params.to_dict(), "features": features, "classes": model.classes,
    "group_accuracy": s["group_accuracy"], "mean_class_accuracy": s["mean_class_accuracy"]}, indent=1))
plot_importance(model.importance()); plt.tight_layout(); plt.show()

## Classify new data
Load new detections in the same way and apply a saved model. New data need the same feature columns. `NEW_GROUP_COL` groups the detections for group predictions (e.g. by encounter or recording); in the outputs, groups are in the `event_id` column.

In [ ]:
RUN_NEW = False
NEW_PATH = "path/to/new/RoccaContourStats.csv"
NEW_GROUP_COL = "event_id"
MODEL_PATH = out / "randomforest_model.joblib"

if RUN_NEW:
    m = RandomForestModel.load(MODEL_PATH)
    new, _ = read_rocca(NEW_PATH, VOC_TYPE, LEVELS, OMIT_VARS) if DATA_TYPE == "rocca" else read_feature_table(NEW_PATH, LABEL_COL, NEW_GROUP_COL)
    det, grp = predict(m, new, group_col=NEW_GROUP_COL, voc_type=VOC_TYPE, min_score=MIN_SCORE)
    det.to_csv(out / "new_detection_predictions.csv", index=False)
    grp.to_csv(out / "new_group_predictions.csv", index=False)
    display(grp)

Classification probabilities through one group, detection by detection:

In [ ]:
if RUN_NEW:
    GROUP = grp["event_id"].iloc[0]
    w = det[det["event_id"] == str(GROUP)].pivot_table(index="detection_id", columns="class", values="probability", sort=False)
    ax = w.reset_index(drop=True).plot(marker="o", ms=3, lw=0.8, figsize=(8, 3))
    ax.set(xlabel="Detection", ylabel="Probability", title=f"{GROUP}"); plt.show()